# Fine-tuning con LoRA en Colab (Fase 5)

Ajusta `Qwen/Qwen2.5-1.5B-Instruct` como clasificador de 3 clases (A gana, B gana, empate) con LoRA.
El texto de entrada es `prompt + respuesta A + respuesta B`. Se ejecuta en Colab con GPU T4
(Entorno de ejecución > Cambiar tipo > T4).

Antes de correrlo, `train.csv` debe estar en la raíz de "Mi unidad". El adaptador, el checkpoint y las
predicciones de validación se guardan en `MyDrive/llm-classification/lora/`.

- Se entrena con `N_TRAIN` filas (la T4 es lenta) y se valida en 3,000 filas separadas **por `prompt`**.
- En cada ejemplo, A y B se intercambian con probabilidad 0.5 (con las etiquetas), igual que en la Fase 2.
- El checkpoint se guarda cada `SAVE_EVERY` pasos: si Colab se desconecta, vuelve a correr todo y retoma.

In [ ]:
!pip -q install -U peft
!pip -q uninstall -y torchao   # Colab ships an old torchao that peft rejects; we do not use it
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DRIVE = Path("/content/drive/MyDrive")
OUT = DRIVE / "llm-classification" / "lora"
OUT.mkdir(parents=True, exist_ok=True)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
N_TRAIN = 20_000       # rows used for training; raise it if the session allows
N_VAL = 3_000
MAX_PROMPT, MAX_RESP = 256, 384   # tokens per field -> at most ~1,024 per example
BATCH, ACCUM = 4, 4               # effective batch 16
LR = 1e-4
SAVE_EVERY = 100
SEED = 42

assert torch.cuda.is_available(), "Activa una GPU: Entorno de ejecución > Cambiar tipo"
print(torch.cuda.get_device_name(0))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
def parse(raw):
    # Same parsing as src/data.py: JSON list of turns, null -> "", turns joined by a blank line.
    return "\n\n".join("" if t is None else t for t in json.loads(raw))


df = pd.read_csv(DRIVE / "train.csv")
assert len(df) == 57_477, f"train.csv has {len(df)} rows (incomplete upload?)"
for col in ["prompt", "response_a", "response_b"]:
    df[col] = df[col].map(parse)
df["label"] = df[["winner_model_a", "winner_model_b", "winner_tie"]].to_numpy().argmax(1)

# Validation split grouped by prompt.
gss = GroupShuffleSplit(n_splits=1, test_size=N_VAL, random_state=SEED)
train_idx, val_idx = next(gss.split(df, groups=df["prompt"]))
train_idx = np.random.default_rng(SEED).permutation(train_idx)[:N_TRAIN]
train, val = df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)
print(len(train), len(val), train["label"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.pad_token or tok.eos_token


def ids(text, limit=None):
    return tok(text, add_special_tokens=False, truncation=limit is not None, max_length=limit)["input_ids"]


HEAD_P, HEAD_A, HEAD_B = ids("<prompt>:\n"), ids("\n\n<response_a>:\n"), ids("\n\n<response_b>:\n")


def encode(prompt, a, b):
    # Each field is truncated separately so that a long A cannot push B out of the context.
    return HEAD_P + ids(prompt, MAX_PROMPT) + HEAD_A + ids(a, MAX_RESP) + HEAD_B + ids(b, MAX_RESP)


def pad_batch(seqs):
    n = max(map(len, seqs))
    x = torch.full((len(seqs), n), tok.pad_token_id)
    mask = torch.zeros((len(seqs), n), dtype=torch.long)
    for i, s in enumerate(seqs):   # right padding; the model reads the last non-pad token
        x[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return x.cuda(), mask.cuda()


SWAP = np.array([1, 0, 2])   # label after swapping A and B

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3, torch_dtype=torch.float16)
model.config.pad_token_id = tok.pad_token_id
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
cfg = LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.05,
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, cfg)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()   # trainable weights (LoRA + score head) in fp32 for stable fp16 autocast
model.cuda()
model.print_trainable_parameters()

In [ ]:
steps_total = (len(train) // BATCH) // ACCUM
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1, (s + 1) / 20) * 0.5 * (1 + math.cos(math.pi * min(s, steps_total) / steps_total)))
scaler = torch.amp.GradScaler()


def trainable_state():
    return {k: v for k, v in model.state_dict().items() if "lora_" in k or "score" in k}


step = 0
ckpt = OUT / "ckpt.pt"
if ckpt.exists():
    state = torch.load(ckpt, map_location="cuda")
    model.load_state_dict(state["model"], strict=False)
    opt.load_state_dict(state["opt"]); sched.load_state_dict(state["sched"]); step = state["step"]
    print("resumed from step", step)
print("optimizer steps:", steps_total)

In [ ]:
model.train()
order = np.random.default_rng(SEED).permutation(len(train))
t0, running = time.time(), []
for s in range(step, steps_total):
    for m in range(ACCUM):
        i = (s * ACCUM + m) * BATCH
        seqs, labels = [], []
        for r in train.iloc[order[i:i + BATCH]].itertuples():
            if random.random() < 0.5:
                seqs.append(encode(r.prompt, r.response_b, r.response_a)); labels.append(SWAP[r.label])
            else:
                seqs.append(encode(r.prompt, r.response_a, r.response_b)); labels.append(r.label)
        x, mask = pad_batch(seqs)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(input_ids=x, attention_mask=mask).logits
        loss = torch.nn.functional.cross_entropy(logits.float(), torch.tensor(labels).cuda()) / ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * ACCUM)
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    step = s + 1
    if step % 10 == 0:
        print(f"step {step}/{steps_total}  loss {np.mean(running[-40:]):.4f}  {(time.time() - t0) / 60:.1f} min")
    if step % SAVE_EVERY == 0 or step == steps_total:
        torch.save({"model": trainable_state(), "opt": opt.state_dict(), "sched": sched.state_dict(), "step": step}, ckpt)

In [ ]:
@torch.no_grad()
def predict(frame, batch=8):
    model.eval()
    out = np.zeros((len(frame), 3))
    # Sort by length so batches pad little; average each row with its A/B-swapped mirror.
    order = np.argsort((frame["response_a"].str.len() + frame["response_b"].str.len()).to_numpy())
    for swap in [False, True]:
        for i in range(0, len(frame), batch):
            idx = order[i:i + batch]
            rows = frame.iloc[idx]
            seqs = [encode(r.prompt, r.response_b, r.response_a) if swap else encode(r.prompt, r.response_a, r.response_b)
                    for r in rows.itertuples()]
            x, mask = pad_batch(seqs)
            with torch.autocast("cuda", dtype=torch.float16):
                p = torch.softmax(model(input_ids=x, attention_mask=mask).logits.float(), -1).cpu().numpy()
            out[idx] += (p[:, SWAP] if swap else p) / 2
    return out


val_pred = predict(val)
print("val log loss:", log_loss(val["label"], val_pred, labels=[0, 1, 2]))
np.savez(OUT / "val_pred.npz", id=val["id"].to_numpy(), pred=val_pred, label=val["label"].to_numpy())

In [ ]:
# Adapter + score head for the Kaggle notebook (the base model is added there as a public Kaggle model).
model.save_pretrained(OUT / "adapter")
tok.save_pretrained(OUT / "adapter")
print(sorted(p.name for p in (OUT / "adapter").iterdir()))